# Scraper to collect the courses for a major at Davidson College

**How to use this notebook:** Click into each code cell and press Shift + Enter to execute the cell. The order matters, so be sure to go top to bottom. 

This cell imports the required modules and set our constants

In [4]:
import requests
from bs4 import BeautifulSoup
import time
import re

Set our constants: the base url that we want to search, some helpful terms/aliases, and our regex helper

In [3]:
BASE_URL = "https://catalog.davidson.edu"
CATALOG_URL = f"{BASE_URL}/preview_program.php?catoid=28&poid=1799" # searches computer science only

CONNECTIVES = {"or", "and", "and/or"}

# Normalize dept aliases so "Math", "Mathematics" → "MAT", "Computer Science" → "CSC", etc.
DEPT_ALIASES = {
    "Mathematics": "MAT",
    "Math": "MAT",
    "Computer Science": "CSC",
    "Physics": "PHY",
    "Biology": "BIO",
    "Digital Studies": "DIG",
}

# Super important! Make sure we don't split the "Not open" strings
NOT_OPEN_PATTERN = re.compile(r'not open to students', re.IGNORECASE)

# Matches "CSC 221", "MAT 150", "Math 112", "Mathematics 150", "Computer Science 121", etc.
COURSE_PATTERN = re.compile(
    r'\b(' + '|'.join(re.escape(k) for k in sorted(DEPT_ALIASES, key=len, reverse=True))
    + r'|[A-Z]{2,4})\s+(\d{3})\b'
)

# Collect all course info

The following cells scrapes all of the course information from the catalog. Note: this includes things like course title, instructor, description, and prerequisites. 

*We do not collect the schedule (e.g., "MWF 12:30-1:20") because that information is not stored in the catalog.*

In [8]:
def get_soup(url):
    """
    Fetches the content of the given URL and returns a BeautifulSoup object.
    """
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")

def get_course_blurb(catoid, coid):
    """
    Fetches the course blurb for a given course identified by catoid and coid,
    which represent the catalog and course IDs respectively.
    """
    url = f"{BASE_URL}/ajax/preview_course.php?catoid={catoid}&coid={coid}&display_options[location]=tooltip&show"
    soup = get_soup(url)
    # The tooltip fragment is a bare <div> or just text — grab all visible text
    return soup.get_text(separator=" ", strip=True)


def normalize_course_code(raw_dept: str, number: str) -> str:
    dept = DEPT_ALIASES.get(raw_dept.strip(), raw_dept.strip().upper())
    return f"{dept} {number}"



def scrape_catalog(catalog_url):
    """
    Scrapes the course catalog page to extract course codes and their blurbs.
    """
    soup = get_soup(catalog_url)
    courses = {} # course_code -> {coid, description} as a dictionary

    # Find all links with the aria-label that starts with "View course details" to identify course links
    for link in soup.find_all("a", attrs={"aria-label": lambda v: v and v.startswith("View course details")}):
        href = link.get("href", "")
        # Only process links that contain "coid=" to ensure we are looking at course detail links
        if "coid=" not in href:
            continue

        # Get necessary parameters from the href to fetch course details
        coid = href.split("coid=")[1].split("&")[0]
        catoid = href.split("catoid=")[1].split("&")[0]
        course_code = link.get_text(strip=True)

        print(f"Fetching {course_code} (coid={coid})...")
        # Save the course code, coid, and description in the courses dictionary
        courses[course_code] = {
            "coid": coid,
            "description": get_course_blurb(catoid, coid),
            "prerequisites": None  # Placeholder for future prerequisite parsing    
        }

        time.sleep(0.5)

    return courses

In [9]:
"""
Cell to start the scraping and save the data to a dictionary called courses.
"""

courses = scrape_catalog(CATALOG_URL)
for code, info in courses.items():
    print(f"\n{code}: {info['description'][:300]}...\nPrerequisites: {info['prerequisites']}")

Fetching CSC 120 (coid=58560)...
Fetching DIG 120 (coid=58503)...
Fetching CSC 121 (coid=57112)...
Fetching CSC 240 (coid=57113)...
Fetching PHY 240 (coid=57673)...
Fetching CSC 209 (coid=57114)...
Fetching BIO 209 (coid=56986)...
Fetching MAT 111 (coid=57443)...
Fetching MAT 112 (coid=57444)...
Fetching MAT 140 (coid=57451)...
Fetching MAT 150 (coid=57452)...
Fetching MAT 230 - Sets and Proofs (coid=57457)...
Fetching CSC 221 - Data Structures (coid=57115)...
Fetching CSC 250 - Computer Organization (coid=58533)...
Fetching CSC 321 - Analysis of Algorithms (coid=57118)...
Fetching CSC 210 Mathematical Modeling (=MAT 210) (coid=59029)...
Fetching CSC 361 Computer Graphics (coid=58424)...
Fetching CSC 362 Data Visualization (coid=58527)...
Fetching CSC 363 - Human Computer Interaction (coid=59460)...
Fetching CSC 370 Machine Reasoning (coid=58431)...
Fetching CSC 371 Machine Learning (coid=58534)...
Fetching CSC 372 - Natural Language Processing (coid=59436)...
Fetching CSC 374 - Deep L

# Parse prerequisites
After we have a dictionary with the information about each course, we can loop over the dictionary to figure out the prerequisites.


In [16]:
NOT_OPEN_PATTERN = re.compile(r'not open to students', re.IGNORECASE)

def parse_prerequisites(prereq_text: str, courses: dict):
    """
    If the prereq text contains course codes that are keys in `courses`,
    return a structured list: ["CSC 121", "or", "CSC 200", ...].
    Otherwise return the raw prereq_text string.
    'Permission of instructor' variants are preserved as a list item if present.
    """
    # Return as raw string for "not open" exclusion notices
    if NOT_OPEN_PATTERN.search(prereq_text):
        return prereq_text

    # Find all course code matches and their positions
    matches = list(COURSE_PATTERN.finditer(prereq_text))
    if not matches:
        return prereq_text

    # Normalize and check if any match a known course
    normalized = [normalize_course_code(m.group(1), m.group(2)) for m in matches]
    if not any(code in courses for code in normalized):
        return prereq_text

    # Build structured list by walking through the text between matches
    result = []
    prev_end = 0

    for i, match in enumerate(matches):
        gap = prereq_text[prev_end:match.start()].strip().rstrip(',').strip()
        if gap:
            gap_words = gap.lower().split()
            for word in gap_words:
                if word in CONNECTIVES:
                    result.append(word)

        code = normalize_course_code(match.group(1), match.group(2))
        result.append(code)
        prev_end = match.end()

    if re.search(r'permission of (?:the )?instructor', prereq_text, re.IGNORECASE):
        result.append("or")
        result.append("Permission of instructor")

    return result

def get_course_prerequisites(blurb: str, courses: dict):
    match = re.search(r'Prerequisite[s]?(?:\s*&\s*Notes)?\s*:?', blurb, re.IGNORECASE)
    if not match:
        return None
    prereq_text = blurb[match.end():].strip()
    return parse_prerequisites(prereq_text, courses)

Confirm that we have our dictionary `courses`

In [17]:
courses

{'CSC 120': {'coid': '58560',
  'description': 'Facebook this Page (opens a new window) Tweet this Page (opens a new window) [ Print Course (opens a new window) ] CSC 120\xa0-\xa0Programming in the Humanities (= DIG 120) Instructor Kabala Computational methods have significantly broadened and deepened the possibilities of inquiry in the Humanities. Programming skills have allowed textual scholars, in particular, to take advantage of enormous digitized corpora of historical documents, newspapers, novels, books, and social network data like Twitter feeds to pose new questions to the written word. We can now trace the changing semantics of words and phrases across millions of documents and hundreds of years, visualize centuries-old plot structures in new ways through sentiment analysis and character networks, and solve long-standing riddles of authorship attribution-among many other exciting feats. This course offers an introduction to computer science through applications in the Humaniti

If `courses` is loaded correctly, then we should be able to parse the description of the courses to get the prerequisites.

In [19]:
# Get the prerequisites for each course after fetching all descriptions to avoid multiple requests per course
for code, info in courses.items():
    print(f"Parsing prerequisites for {code}...")
    info["prerequisites"] = get_course_prerequisites(info["description"], courses)
    #time.sleep(0.5)

Parsing prerequisites for CSC 120...
Parsing prerequisites for DIG 120...
Parsing prerequisites for CSC 121...
Parsing prerequisites for CSC 240...
Parsing prerequisites for PHY 240...
Parsing prerequisites for CSC 209...
Parsing prerequisites for BIO 209...
Parsing prerequisites for MAT 111...
Parsing prerequisites for MAT 112...
Parsing prerequisites for MAT 140...
Parsing prerequisites for MAT 150...
Parsing prerequisites for MAT 230 - Sets and Proofs...
Parsing prerequisites for CSC 221 - Data Structures...
Parsing prerequisites for CSC 250 - Computer Organization...
Parsing prerequisites for CSC 321 - Analysis of Algorithms...
Parsing prerequisites for CSC 210 Mathematical Modeling (=MAT 210)...
Parsing prerequisites for CSC 361 Computer Graphics...
Parsing prerequisites for CSC 362 Data Visualization...
Parsing prerequisites for CSC 363 - Human Computer Interaction...
Parsing prerequisites for CSC 370 Machine Reasoning...
Parsing prerequisites for CSC 371 Machine Learning...
Pars

In [20]:
courses

{'CSC 120': {'coid': '58560',
  'description': 'Facebook this Page (opens a new window) Tweet this Page (opens a new window) [ Print Course (opens a new window) ] CSC 120\xa0-\xa0Programming in the Humanities (= DIG 120) Instructor Kabala Computational methods have significantly broadened and deepened the possibilities of inquiry in the Humanities. Programming skills have allowed textual scholars, in particular, to take advantage of enormous digitized corpora of historical documents, newspapers, novels, books, and social network data like Twitter feeds to pose new questions to the written word. We can now trace the changing semantics of words and phrases across millions of documents and hundreds of years, visualize centuries-old plot structures in new ways through sentiment analysis and character networks, and solve long-standing riddles of authorship attribution-among many other exciting feats. This course offers an introduction to computer science through applications in the Humaniti